# Phase 4 — Explanation Readability Assessment (`lawgic-tos-changes`)

**Claim under test.** The application's value proposition is that it makes Terms of Service
*readable*. That has never been measured — only asserted. This notebook measures it, using
the evaluation approach of **Pravasi & Das (2024)**, who scored ChatGPT interpretations of
privacy policies with Flesch readability metrics.

## Which application

The thesis artifact is **`lawgic-tos-changes`** (Next.js, ToS version-diff flow).
`lawgic-web-app` is deprecated and is not measured here.

That distinction changes the design. `lawgic-web-app` explained **one clause at a time**
via the FastAPI `/api/explain_tos_scores` endpoint. `lawgic-tos-changes` has no per-clause
explain flow at all: it diffs **two versions** of a document and emits a list of *changes*,
each carrying its own plain-language fields. So the paired unit is one **change**, not one
clause:

| Side of the pair | Field | What it is |
| --- | --- | --- |
| Source (legalese) | `new_text`, or `old_text` when the change is a removal | The excerpt the model quotes verbatim from the actual ToS, <=400 chars |
| Explanation (plain language) | `what_changed` + `impact_for_user` | Exactly the prose a user reads on the change card |

## Fidelity

The notebook does not re-implement the prompt. It drives the **running application's own
HTTP endpoints**, so `buildChunkPrompt()` in `lib/ollama-diff.js` is executed by the app
itself, byte for byte, including its system prompt, its `gemma4:31b-cloud` call through
`/v1/chat/completions`, its `cleanJsonContent()` fence handling and its
retry-once-on-`SyntaxError` behaviour.

The client-side orchestration in `pages/index.js` is replicated exactly, including two
details that are easy to miss and would change the sample if skipped:

1. **The Legal-BERT context reaches only the first chunk.** `index.js:146` passes
   `lawgicClauses` when `batchStart + indexInBatch === 0` and `null` otherwise. Feeding it
   to every chunk would be a different system.
2. **The change list is deduplicated client-side** by `mergeAndDedupeChanges()` before the
   user sees it. Scoring the raw per-chunk output would double-count near-identical changes
   found in overlapping chunks. That function is ported to Python below, with a self-check.

## Caveat to carry into the manuscript

In this app **`harm_label` is assigned by the LLM, not by Legal-BERT.** The classifier's
output enters the prompt as advisory context (`buildLawgicContext`, truncated to 2,000
chars per version) — it does not set the label. So the per-risk-class breakdown below
stratifies by *the generator's own* risk judgement. Do not describe it as the classifier's
verdict.

Outputs (in `generated_files/lawgic_taxonomy/evaluation/`):
- `phase4_changes.jsonl` — raw generated changes (the resume checkpoint)
- `phase4_readability_pairs.csv` — one row per change with all scores
- `phase4_readability_analysis.csv`, `phase4_readability_summary.{csv,tex}`
- `phase4_readability.png`

## MANUAL STEP — before running anything below

1. **Install the readability library** (the one Pravasi & Das used):
   ```bash
   /Users/riki/anaconda3/envs/thesis-env/bin/pip install py-readability-metrics
   /Users/riki/anaconda3/envs/thesis-env/bin/python -m nltk.downloader punkt
   ```

2. **Start the application.** This notebook is a client; it does nothing without the app.
   ```bash
   cd "/Users/riki/Coding Projects/Thesis/lawgic-tos-changes"
   npm run dev          # http://localhost:3000
   ```
   `lawgic-tos-changes/.env.local` must have `OLLAMA_MODEL=gemma4:31b-cloud` (it does) and
   the machine needs internet plus a signed-in Ollama session.

3. **Optional but recommended — start the classifier backend**, so the Legal-BERT context
   actually reaches the prompt:
   ```bash
   cd "/Users/riki/Coding Projects/Thesis/lawgic"
   /Users/riki/anaconda3/envs/thesis-env/bin/python3 -m uvicorn api.server:app --port 8000
   ```
   The app degrades gracefully without it (`pages/api/analyze.js` returns `skipped: true`
   and `lawgicClauses` becomes `null`). The preflight cell reports which mode you are in —
   **record it**, because it changes what the prompt contained.

4. **Confirm Ollama is reachable:**
   ```bash
   curl -s http://localhost:11434/api/tags >/dev/null && echo ok
   ```

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    sentinel = Path("generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv")
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import lawgic_eval_core as core

APP_URL = "http://localhost:3000"          # Next.js dev server (lawgic-tos-changes)
CLASSIFIER_URL = "http://localhost:8000"   # FastAPI, optional
CHUNK_TIMEOUT = 300                        # matches maxDuration in pages/api/diff/chunk.js

# lib/constants.js — mirrored so the run matches the app exactly.
CASE_STUDY_SERVICES = [
    {"id": "tiktok", "label": "TikTok"},
    {"id": "youtube", "label": "YouTube"},
    {"id": "x", "label": "X (Twitter)"},
]

core.EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)


def preflight() -> bool:
    try:
        requests.get(f"{APP_URL}/api/case-studies/tiktok", timeout=30).raise_for_status()
        print(f"lawgic-tos-changes  : UP at {APP_URL}")
    except Exception as exc:
        raise SystemExit(f"App not reachable at {APP_URL} — run `npm run dev`. ({exc})")

    try:
        requests.get("http://localhost:11434/api/tags", timeout=10).raise_for_status()
        print("ollama              : UP")
    except Exception as exc:
        raise SystemExit(f"Ollama not reachable ({exc})")

    try:
        requests.get(f"{CLASSIFIER_URL}/docs", timeout=10)
        print(f"Legal-BERT backend  : UP at {CLASSIFIER_URL} — classifier context WILL be injected")
        return True
    except Exception:
        print("Legal-BERT backend  : DOWN — lawgicClauses will be null. Record this.")
        return False


CLASSIFIER_AVAILABLE = preflight()

## Fixed user profile

`buildChunkPrompt` personalises `impact_for_user` to the reader's role and concerns, so the
profile is an input variable, not a constant of the system. One profile is fixed for the
whole run and recorded on every row.

`creator` is chosen because it is the role with the most concern options in
`lib/constants.js` (five), which exercises the personalisation path rather than leaving it
mostly empty. **Readability may differ by role** — a real limitation of a single-profile
design, and a second run under a different profile is the way to test it. State that in the
manuscript rather than implying the result is role-invariant.

In [ ]:
PROFILE = {
    "roleLabel": "Content Creator",
    "concerns": ["I'm monetized on this platform", "I post original music or audio"],
    "context": "I run a monetized channel and post original music.",
}
print(json.dumps(PROFILE, indent=2))

## Port of `mergeAndDedupeChanges` (`lib/merge-diff-results.js`)

Faithful port: same text normalisation, same >2-character word filter, same Jaccard
threshold of 0.82, same harm-then-type sort order, same `change_NNN` renumbering. Both
JavaScript's `Array.prototype.sort` and Python's `sorted` are stable, so tied elements keep
insertion order in either language.

The self-check asserts the three behaviours that matter: exact duplicates collapse,
near-duplicates above the threshold collapse, and the sort puts harmful before neutral
before fair.

In [ ]:
HARM_ORDER = {"harmful": 0, "neutral": 1, "fair": 2}
TYPE_ORDER = {"modified": 0, "added": 1, "removed": 2}


def normalize_change_text(value: str = "") -> str:
    text = re.sub(r"[^a-z0-9\s]+", " ", str(value or "").lower())
    return re.sub(r"\s+", " ", text).strip()


def word_set(value: str) -> set:
    return {w for w in normalize_change_text(value).split(" ") if len(w) > 2}


def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    intersection = len(a & b)
    return intersection / (len(a) + len(b) - intersection)


def merge_and_dedupe_changes(changes: list[dict]) -> list[dict]:
    seen_normalized, seen_word_sets, deduped = set(), [], []
    for change in changes:
        if not change:
            continue
        normalized = normalize_change_text(change.get("what_changed"))
        if normalized:
            if normalized in seen_normalized:
                continue
            words = word_set(normalized)
            if any(jaccard(words, existing) >= 0.82 for existing in seen_word_sets):
                continue
            seen_normalized.add(normalized)
            seen_word_sets.append(words)
        deduped.append(change)

    deduped.sort(key=lambda c: (HARM_ORDER.get(c.get("harm_label"), 9),
                                TYPE_ORDER.get(c.get("change_type"), 9)))
    return [{**c, "id": f"change_{i + 1:03d}"} for i, c in enumerate(deduped)]


def _dedupe_self_check() -> None:
    sample = [
        {"what_changed": "The service may now collect device location data",
         "harm_label": "fair", "change_type": "added"},
        {"what_changed": "The service may now collect device location data",
         "harm_label": "harmful", "change_type": "modified"},        # exact duplicate
        {"what_changed": "The service may now collect device location data too",
         "harm_label": "harmful", "change_type": "added"},           # near duplicate
        {"what_changed": "Arbitration is now mandatory for all disputes",
         "harm_label": "harmful", "change_type": "modified"},
    ]
    result = merge_and_dedupe_changes(sample)
    assert len(result) == 2, [r["what_changed"] for r in result]
    assert result[0]["harm_label"] == "harmful", result[0]
    assert result[0]["id"] == "change_001" and result[1]["id"] == "change_002"
    assert jaccard(word_set("alpha beta gamma"), word_set("alpha beta gamma")) == 1.0
    print("dedupe self-check: exact + near duplicates collapse, harm ordering holds")


_dedupe_self_check()

## Run the diff flow over the three case studies

Replicates `pages/index.js` step for step:

1. `POST /api/analyze` twice (old and new) to obtain `lawgicClauses`, skipped if the
   classifier backend is down — same `try/catch -> null` fallback as the app.
2. `POST /api/diff/prepare` to chunk both documents into aligned section pairs.
3. `POST /api/diff/chunk` per pair, passing `lawgicClauses` **only on the first chunk** and
   `null` thereafter.
4. `mergeAndDedupeChanges` over everything collected for that document.

The app fires chunks two at a time (`DIFF_CHUNK_CONCURRENCY`); this runs them sequentially.
Concurrency changes throughput, not prompts or outputs — each chunk call is independent.

**Resumable:** each service's changes are appended to `phase4_changes.jsonl` as soon as that
service completes. Re-running skips services already present. Delete the file to start over.

Expect a few minutes per document: 25–58 KB of text at `DIFF_CHUNK_MAX_CHARS = 5500` is
roughly 5–11 chunks each, at a few seconds per chunk call.

In [ ]:
FORCE_RERUN = False
changes_path = core.EVAL_OUT_DIR / "phase4_changes.jsonl"

completed: dict[str, list] = {}
if changes_path.exists() and not FORCE_RERUN:
    for line in changes_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            completed.setdefault(record["service_id"], []).append(record)
    print(f"Resuming: {sorted(completed)} already generated.")


def post_json(path: str, payload: dict, timeout: int = 120) -> dict:
    response = requests.post(f"{APP_URL}{path}", json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()


def classify(text: str, label: str):
    """POST /api/analyze — the same call index.js makes. Returns clauses or None."""
    if not CLASSIFIER_AVAILABLE:
        return None
    try:
        response = requests.post(
            f"{APP_URL}/api/analyze",
            files={"file": ("tos.txt", text, "text/plain")},
            data={"service_name": label},
            timeout=300,
        )
        response.raise_for_status()
        data = response.json()
        return None if data.get("skipped") else data.get("clauses")
    except Exception as exc:
        print(f"    classifier failed for {label}: {str(exc)[:120]} -> lawgicClauses=None")
        return None


def run_service(service: dict) -> list[dict]:
    study = requests.get(f"{APP_URL}/api/case-studies/{service['id']}", timeout=60).json()
    old_tos, new_tos = study["oldTos"], study["newTos"]
    old_label = study.get("oldLabel", "Old version")
    new_label = study.get("newLabel", "New version")
    print(f"  {service['label']}: {len(old_tos):,} -> {len(new_tos):,} chars")

    lawgic_old = classify(old_tos, old_label)
    lawgic_new = classify(new_tos, new_label)
    print(f"    lawgicClauses: old={'set' if lawgic_old else 'null'} new={'set' if lawgic_new else 'null'}")

    prepared = post_json("/api/diff/prepare", {
        "oldTos": old_tos, "newTos": new_tos, "oldLabel": old_label, "newLabel": new_label,
    })
    pairs, total = prepared["pairs"], prepared["totalPairs"]
    print(f"    {total} section pairs")

    collected = []
    for index, pair in enumerate(pairs):
        try:
            result = post_json("/api/diff/chunk", {
                "oldText": pair.get("oldText", ""),
                "newText": pair.get("newText", ""),
                "oldLabel": old_label,
                "newLabel": new_label,
                "sectionTitle": pair.get("sectionTitle") or pair.get("title"),
                "sectionIndex": pair.get("sectionIndex"),
                "totalSections": pair.get("totalSections"),
                "serviceLabel": service["label"],
                "profile": PROFILE,
                # index.js:146 — context goes to the first chunk only.
                "lawgicClauses": {"old": lawgic_old, "new": lawgic_new} if index == 0 else None,
            }, timeout=CHUNK_TIMEOUT)
            collected.extend(result.get("changes", []))
        except Exception as exc:
            print(f"    section {index + 1}/{total} FAILED: {str(exc)[:120]}")
        if (index + 1) % 2 == 0 or index + 1 == total:
            print(f"    {index + 1}/{total} sections | {len(collected)} raw changes")

    merged = merge_and_dedupe_changes(collected)
    print(f"    {len(collected)} raw -> {len(merged)} after dedupe")
    return [{
        **change,
        "service_id": service["id"],
        "service_label": service["label"],
        "old_label": old_label,
        "new_label": new_label,
        "classifier_context": bool(lawgic_old or lawgic_new),
        "profile_role": PROFILE["roleLabel"],
    } for change in merged]


started = time.time()
with changes_path.open("a", encoding="utf-8") as sink:
    for service in CASE_STUDY_SERVICES:
        if service["id"] in completed:
            print(f"skip {service['label']} (already generated)")
            continue
        print(f"\nrunning {service['label']} ...")
        records = run_service(service)
        for record in records:
            sink.write(json.dumps(record) + "\n")
        sink.flush()
        completed[service["id"]] = records

all_changes = [record for records in completed.values() for record in records]
print(f"\nTotal changes: {len(all_changes)} across {len(completed)} services "
      f"({(time.time() - started) / 60:.1f} min this session)")

## Build the readability pairs

**Source side.** `new_text` is the legalese excerpt for `modified` and `added` changes. For
`removed` changes `new_text` is empty by schema, so `old_text` is used instead. Changes with
neither are dropped and counted.

**Explanation side.** `what_changed` + `impact_for_user` joined with a space — together they
are the prose body of one change card. `action_needed` is excluded: it is an imperative
instruction, not an explanation of the clause, and would inflate readability with short
command sentences.

In [ ]:
rows = []
dropped_no_text = 0
for change in all_changes:
    source = (change.get("new_text") or "").strip() or (change.get("old_text") or "").strip()
    explanation = " ".join(
        part for part in (change.get("what_changed"), change.get("impact_for_user")) if part
    ).strip()
    if not source or not explanation:
        dropped_no_text += 1
        continue
    rows.append({
        "id": change.get("id"),
        "service": change.get("service_label"),
        "change_type": change.get("change_type"),
        "harm_label": change.get("harm_label"),
        "topic": change.get("topic"),
        "clause_text": source,
        "explanation_text": explanation,
        "source_side": "new_text" if (change.get("new_text") or "").strip() else "old_text",
        "classifier_context": change.get("classifier_context"),
        "profile_role": change.get("profile_role"),
        "model": "gemma4:31b-cloud",
    })

pairs_frame = pd.DataFrame(rows)
print(f"Usable pairs: {len(pairs_frame)} (dropped {dropped_no_text} with no quotable text)")
print("\nBy generator-assigned harm label:")
print(pairs_frame["harm_label"].value_counts().to_string())
print("\nBy service:")
print(pairs_frame["service"].value_counts().to_string())
display(pairs_frame.head())

## Readability scoring

`py-readability-metrics` is the library used by Pravasi & Das, so it is the primary
instrument. It **requires at least 100 words** and raises `ReadabilityException` below that.
ToS excerpts are capped at 400 characters by the prompt schema and explanations are two or
three sentences, so **most pairs will fall under the gate** — this cell records exactly how
many and why. A pair is excluded if *either* side fails; a paired test cannot use a
half-scored pair.

Because that gate can remove nearly everything, a clearly-labelled **secondary** column set
recomputes the same two Flesch formulas inline (standard definitions, vowel-group syllable
heuristic) with no length gate. Report the library rows as the Pravasi & Das replication;
cite the all-pairs rows as a robustness check, never as the same measurement — the syllable
counts are heuristic and will differ slightly from the library's.

In [ ]:
from readability import Readability
from readability.exceptions import ReadabilityException


def library_scores(text: str) -> dict:
    try:
        readability = Readability(text)
        return {
            "flesch_ease": float(readability.flesch().score),
            "flesch_kincaid": float(readability.flesch_kincaid().score),
            "exclusion_reason": None,
        }
    except ReadabilityException as exc:
        return {"flesch_ease": np.nan, "flesch_kincaid": np.nan, "exclusion_reason": str(exc)}


# ponytail: heuristic syllable counter, only because py-readability-metrics refuses texts
# under 100 words and the prompt caps quoted excerpts at 400 chars. The *_lib columns are
# the real instrument wherever a text is long enough to qualify.
VOWEL_GROUPS = re.compile(r"[aeiouy]+")


def count_syllables(word: str) -> int:
    word = word.lower().strip(".,;:!?\"'()[]")
    if not word:
        return 0
    syllables = len(VOWEL_GROUPS.findall(word))
    if word.endswith("e") and syllables > 1 and not word.endswith(("le", "ee")):
        syllables -= 1
    return max(syllables, 1)


def fallback_scores(text: str) -> dict:
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    words = re.findall(r"[A-Za-z']+", text)
    if not words or not sentences:
        return {"flesch_ease_raw": np.nan, "flesch_kincaid_raw": np.nan, "words": 0, "sentences": 0}
    words_per_sentence = len(words) / len(sentences)
    syllables_per_word = sum(count_syllables(w) for w in words) / len(words)
    return {
        "flesch_ease_raw": 206.835 - 1.015 * words_per_sentence - 84.6 * syllables_per_word,
        "flesch_kincaid_raw": 0.39 * words_per_sentence + 11.8 * syllables_per_word - 15.59,
        "words": len(words),
        "sentences": len(sentences),
    }


scored = pairs_frame.copy()
for side in ("clause", "explanation"):
    texts = scored[f"{side}_text"]
    lib = texts.map(library_scores).apply(pd.Series)
    scored[f"{side}_flesch_ease_lib"] = lib["flesch_ease"]
    scored[f"{side}_flesch_kincaid_lib"] = lib["flesch_kincaid"]
    scored[f"{side}_exclusion"] = lib["exclusion_reason"]
    raw = texts.map(fallback_scores).apply(pd.Series)
    for column in raw.columns:
        scored[f"{side}_{column}"] = raw[column]

scored["library_scored"] = scored["clause_exclusion"].isna() & scored["explanation_exclusion"].isna()
scored.to_csv(core.EVAL_OUT_DIR / "phase4_readability_pairs.csv", index=False)

n_total, n_lib = len(scored), int(scored["library_scored"].sum())
print(f"Pairs                              : {n_total}")
print(f"Scorable by py-readability-metrics : {n_lib}")
print(f"Excluded for length (<100 words)   : {n_total - n_lib} ({(n_total - n_lib) / max(n_total, 1):.1%})")
print(f"  clause too short                 : {int(scored['clause_exclusion'].notna().sum())}")
print(f"  explanation too short            : {int(scored['explanation_exclusion'].notna().sum())}")
print(f"\nMedian words - clause {scored['clause_words'].median():.0f}, "
      f"explanation {scored['explanation_words'].median():.0f}")

## Paired analysis

Procedure decided by the data, not by preference:

1. Shapiro–Wilk on the paired differences. Normality not rejected at alpha = 0.05 →
   **paired t-test** with **Cohen's d for paired samples**; otherwise **Wilcoxon
   signed-rank** with the **matched-pairs rank-biserial correlation**.
2. Proportion of pairs whose readability improved. The two metrics run in opposite
   directions: higher Flesch Reading Ease is easier, lower Flesch-Kincaid Grade is easier.
3. Breakdown by the generator-assigned `harm_label`, and by service.

In [ ]:
from scipy import stats

ALPHA = 0.05


def paired_test(clause_values, explanation_values, higher_is_better: bool) -> dict:
    clause_values = np.asarray(clause_values, dtype=float)
    explanation_values = np.asarray(explanation_values, dtype=float)
    keep = ~(np.isnan(clause_values) | np.isnan(explanation_values))
    clause_values, explanation_values = clause_values[keep], explanation_values[keep]
    n = len(clause_values)
    if n < 3:
        return {"n": n, "test": "insufficient data"}

    differences = explanation_values - clause_values
    normal = bool(stats.shapiro(differences).pvalue > ALPHA) if 3 <= n <= 5000 else False

    if normal:
        statistic, p_value = stats.ttest_rel(explanation_values, clause_values)
        effect_name = "cohens_d_paired"
        effect = float(differences.mean() / differences.std(ddof=1))
        test_name = "paired t-test"
    else:
        result = stats.wilcoxon(explanation_values, clause_values)
        statistic, p_value = result.statistic, result.pvalue
        nonzero = differences[differences != 0]
        ranks = stats.rankdata(np.abs(nonzero))
        signs = np.sign(nonzero)
        total = ranks.sum()
        effect_name = "rank_biserial"
        effect = float((ranks[signs > 0].sum() - ranks[signs < 0].sum()) / total) if total else 0.0
        test_name = "Wilcoxon signed-rank"

    improved = differences > 0 if higher_is_better else differences < 0
    return {
        "n": n, "test": test_name, "shapiro_normal": normal,
        "clause_mean": float(clause_values.mean()),
        "explanation_mean": float(explanation_values.mean()),
        "mean_difference": float(differences.mean()),
        "statistic": float(statistic), "p_value": float(p_value),
        "effect_size_name": effect_name, "effect_size": effect,
        "proportion_improved": float(improved.mean()),
    }


METRICS = [
    ("Flesch Reading Ease (library)", "clause_flesch_ease_lib", "explanation_flesch_ease_lib", True),
    ("Flesch-Kincaid Grade (library)", "clause_flesch_kincaid_lib", "explanation_flesch_kincaid_lib", False),
    ("Flesch Reading Ease (all pairs)", "clause_flesch_ease_raw", "explanation_flesch_ease_raw", True),
    ("Flesch-Kincaid Grade (all pairs)", "clause_flesch_kincaid_raw", "explanation_flesch_kincaid_raw", False),
]

strata = [("all", scored)]
strata += [(f"harm={label}", group) for label, group in scored.groupby("harm_label")]
strata += [(f"service={name}", group) for name, group in scored.groupby("service")]

results = [
    {"Metric": label, "Stratum": stratum,
     **paired_test(subset[clause_col], subset[explanation_col], higher_is_better)}
    for stratum, subset in strata
    for label, clause_col, explanation_col, higher_is_better in METRICS
]

analysis = pd.DataFrame(results)
analysis.to_csv(core.EVAL_OUT_DIR / "phase4_readability_analysis.csv", index=False)
display(analysis[analysis["Stratum"] == "all"])
display(analysis[analysis["Stratum"] != "all"])

In [ ]:
summary = analysis[analysis["Stratum"] == "all"].copy()
summary_table = pd.DataFrame({
    "Metric": summary["Metric"],
    "n": summary["n"],
    "Clause": summary["clause_mean"].round(2),
    "Explanation": summary["explanation_mean"].round(2),
    "Difference": summary["mean_difference"].round(2),
    "Test": summary["test"],
    "p": summary["p_value"].map(lambda p: f"{p:.2e}" if p < 0.001 else f"{p:.3f}"),
    "Effect": summary.apply(lambda r: f"{r['effect_size']:.2f} ({r['effect_size_name']})", axis=1),
    "Improved": summary["proportion_improved"].map(lambda v: f"{v:.1%}"),
})
display(summary_table)

core.write_outputs(
    summary_table,
    "phase4_readability_summary",
    caption=(
        f"Readability of quoted Terms of Service excerpts versus the plain-language change "
        f"explanations generated by lawgic-tos-changes (gemma4:31b-cloud), over {len(scored)} "
        f"changes from {scored['service'].nunique()} ToS version pairs. Library rows use "
        "py-readability-metrics (the instrument of Pravasi \\& Das, 2024), which requires at "
        "least 100 words and therefore covers only the longer pairs; the all-pairs rows "
        "recompute the same Flesch formulas without a length gate as a robustness check. "
        "Higher Reading Ease and lower Grade Level indicate easier text."
    ),
    label="tab:readability",
)

## Figure — paired distributions

Left: violin plot of the two distributions. Right: a slope plot, one line per change, so the
*pairing* is visible — the claim is about the direction of those lines, not merely the gap
between two means.

In [ ]:
clause_col, explanation_col, metric_label = (
    "clause_flesch_ease_raw", "explanation_flesch_ease_raw", "Flesch Reading Ease"
)

plot_data = scored[[clause_col, explanation_col, "harm_label"]].dropna()
figure, (violin_axis, slope_axis) = plt.subplots(1, 2, figsize=(12, 5))

violin_axis.violinplot([plot_data[clause_col], plot_data[explanation_col]], showmedians=True)
violin_axis.set_xticks([1, 2])
violin_axis.set_xticklabels(["ToS excerpt", "Explanation"])
violin_axis.set_ylabel(metric_label)
violin_axis.set_title(f"{metric_label} distribution (n={len(plot_data)})")
violin_axis.grid(axis="y", alpha=0.3)

colours = {"harmful": "#c0392b", "neutral": "#7f8c8d", "fair": "#27ae60"}
for _, row in plot_data.iterrows():
    slope_axis.plot([0, 1], [row[clause_col], row[explanation_col]],
                    color=colours.get(row["harm_label"], "#999999"), alpha=0.35, linewidth=0.9)
slope_axis.set_xticks([0, 1])
slope_axis.set_xticklabels(["ToS excerpt", "Explanation"])
slope_axis.set_ylabel(metric_label)
slope_axis.set_title("Paired change per item")
slope_axis.grid(axis="y", alpha=0.3)
slope_axis.legend(handles=[plt.Line2D([], [], color=c, label=k) for k, c in colours.items()],
                  title="harm_label (LLM-assigned)", loc="best", fontsize=8)

figure.suptitle("ToS excerpts vs generated change explanations (lawgic-tos-changes, gemma4:31b-cloud)")
figure.tight_layout()

figure_path = core.EVAL_OUT_DIR / "phase4_readability.png"
figure.savefig(figure_path, dpi=200, bbox_inches="tight")
print(f"Wrote {figure_path}")
plt.show()